# AI Model Experiment & Evaluation
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membandingkan dua pendekatan AI dalam menyelesaikan task sentiment analysis ulasan pelanggan:
1. Model klasik Machine Learning (Scikit-learn)
2. LLM API (Gemini)

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

> 🔧 **Catatan Penyesuaian Dataset**
>
> Notebook ini menggunakan dataset `customer_reviews_sentiment.csv` dengan kolom `review_text` dan `sentiment` (nilai: `positif`/`negatif`).
>
> Apabila dataset final berbeda dari yang digunakan saat ini, sesuaikan bagian berikut:
> - Nama file pada `pd.read_csv(...)` di Section 2
> - Nama kolom teks dan label pada Section 3 (saat ini: `review_text`, `sentiment`)
> - Label yang diminta pada prompt LLM di Section 5.2 (saat ini: `positif`/`negatif`)
> - Studi Kasus pada Assignment Brief, apabila domain data berbeda dari e-commerce

## 1. Problem Statement

**Konteks.** Tim produk e-commerce ingin menambahkan fitur yang otomatis memberi label sentimen pada ulasan pelanggan di halaman produk. Sebelum memilih pendekatan untuk produksi, dua kandidat dibandingkan pada data dan test set yang sama:
1. **Model klasik**: TF-IDF + Logistic Regression (Scikit-learn), dilatih dari dataset ulasan.
2. **LLM API**: Gemini dengan prompt zero-shot, tanpa proses training.

**Objective.** Mengukur performa kedua pendekatan (Accuracy, Precision, Recall, F1-Score, confusion matrix), membandingkan effort implementasi, kecepatan, dan biaya, lalu menentukan pendekatan mana yang layak dikembangkan lebih lanjut.

**Target/label.** Kolom `sentiment`, klasifikasi biner:
- `positif`: ulasan yang menunjukkan pelanggan puas, misalnya produk sesuai deskripsi, kualitas baik, pengiriman cepat, penjual ramah.
- `negatif`: ulasan yang menunjukkan pelanggan kecewa, misalnya barang rusak atau tidak sesuai, pengiriman lambat, penjual tidak responsif.

Input untuk kedua model hanya kolom `review_text`.

**Batasan dan asumsi.**
- Dataset dipakai apa adanya (200 baris). File CSV asli tidak diubah; semua pengolahan dilakukan di memori.
- Hanya ada dua kelas. Tidak ada kelas netral, jadi ulasan yang bernada campuran tetap harus masuk ke salah satu label.
- Label pada dataset dianggap ground truth yang benar.
- `product_name` dan `review_id` tidak dipakai sebagai fitur. `review_id` hanya dipakai sebagai identitas sampel.
- `positif` diperlakukan sebagai kelas positif untuk Precision/Recall/F1 binary. Macro-average ditampilkan sebagai pembanding.
- Gemini dipakai secara zero-shot, tanpa fine-tuning dan tanpa contoh dari training set.
- Training model deep learning dari nol berada di luar scope assignment.
- Dataset hanya berisi 40 teks unik (lihat Section 2), sehingga split utama mengandung teks yang sama di train dan test. Karena itu ditambahkan evaluasi pendukung berbasis group split (Section 3).

## 2. Import Library & Load Dataset

In [ ]:
# 🔧 Sesuaikan nama file jika dataset final berbeda
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_recall_fscore_support)
from google import genai  # SDK resmi: google-genai
from google.genai import types

# Notebook dijalankan dari folder notebook/, jadi path data relatif ke folder tersebut
DATA_PATH = '../data/customer_reviews_sentiment.csv'
DOC_DIR = '../documentation'
os.makedirs(DOC_DIR, exist_ok=True)

LABELS = ['negatif', 'positif']   # urutan label untuk confusion matrix
POS_LABEL = 'positif'             # kelas positif untuk metrik binary
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
df.head()

Eksplorasi singkat: ukuran data, missing values, distribusi label, dan panjang review.

In [ ]:
print('Ukuran dataset (baris, kolom):', df.shape)
print('\nMissing values per kolom:')
print(df.isna().sum())

label_dist = df['sentiment'].value_counts().to_frame('jumlah')
label_dist['persen'] = (label_dist['jumlah'] / len(df) * 100).round(1)
print('\nDistribusi label:')
print(label_dist)

print('\nPanjang review (jumlah kata):')
print(df['review_text'].str.split().str.len().describe().round(2))

Cek duplikasi teks. Ini penting karena teks yang sama bisa masuk ke train dan test sekaligus.

In [ ]:
label_per_text = df.groupby('review_text')['sentiment'].nunique()

print('Jumlah baris                      :', len(df))
print('Jumlah teks unik                  :', df['review_text'].nunique())
print('Baris yang teksnya duplikat       :', df['review_text'].duplicated().sum())
print('Teks identik dengan label berbeda :', (label_per_text > 1).sum())
print('\nTeks unik per label:')
print(df.drop_duplicates('review_text')['sentiment'].value_counts())

print('\n10 teks yang paling sering muncul:')
df['review_text'].value_counts().head(10).to_frame('frekuensi')

**Catatan dari eksplorasi data**
- Dataset berisi 200 ulasan tanpa missing values. Label cukup seimbang: 110 positif (55%) dan 90 negatif (45%).
- Ulasannya pendek, 6–11 kata.
- Hanya ada **40 teks unik** (20 positif, 20 negatif). Setiap teks muncul 1–12 kali dengan `review_id` dan `product_name` yang berbeda.
- Tidak ada teks identik dengan label berbeda, jadi labelnya konsisten.
- Konsekuensinya, split acak per baris hampir pasti menaruh teks yang sama di train dan test. Dampaknya diukur di Section 3.

## 3. Menyiapkan Train/Test Split

Pastikan test set yang sama digunakan untuk kedua pendekatan agar perbandingan adil.

In [ ]:
# 🔧 Sesuaikan nama kolom jika dataset final menggunakan nama kolom berbeda
X = df['review_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

Split di atas adalah **split utama** dari starter notebook dan dipakai untuk kedua pendekatan. `X_test` dan `y_test` yang sama akan dikirim ke Gemini di Section 5. `review_id` sampel test disimpan sebagai identitas agar urutan dan isi test set bisa dicek ulang.

In [ ]:
test_ids_main = df.loc[X_test.index, 'review_id'].to_numpy()

dist_main = pd.DataFrame({
    'train': y_train.value_counts(),
    'test': y_test.value_counts(),
}).loc[LABELS]
dist_main.loc['total'] = dist_main.sum()
print('Distribusi label split utama:')
print(dist_main)
print('\nreview_id test set (10 pertama):', test_ids_main[:10])

Cek seberapa banyak teks test yang juga muncul di training set:

In [ ]:
test_in_train = X_test.isin(set(X_train))
print('Teks unik di test set                      :', X_test.nunique())
print(f'Baris test yang teksnya juga ada di train  : {test_in_train.sum()} dari {len(X_test)}')
print('Teks test yang tidak pernah muncul di train:', X_test[~test_in_train].tolist())

**Implikasi untuk evaluasi.** Pada split utama, hampir semua review test (39 dari 40) punya kembaran persis di training set. Model Scikit-learn diuji pada teks yang sudah pernah ia lihat saat training, sehingga skornya **tidak bisa dibaca sebagai kemampuan generalisasi** ke review baru. Gemini tidak memakai data training, jadi tidak diuntungkan oleh kebocoran ini.

Split utama tetap dipakai sebagai eksperimen utama sesuai starter notebook. Sebagai pembanding, dibuat dua analisis tambahan:
1. **Evaluasi pada 26 teks unik di test set utama.** Analisis ini hanya menghilangkan efek frekuensi duplikasi (teks yang muncul berkali-kali tidak dihitung berkali-kali). Teks tersebut tetap pernah dilihat model saat training, jadi **ini bukan evaluasi bebas leakage**.
2. **Group-based split berdasarkan `review_text`.** Teks yang identik dijamin hanya ada di salah satu sisi, train atau test. Kedua pendekatan dievaluasi pada test set grup yang sama.

#### Group-based split (evaluasi tambahan)

Metode: `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` dengan `groups = review_text`. Dengan 5 fold, satu fold berisi sekitar 20% data, setara dengan `test_size=0.2` pada split utama. **Fold pertama (indeks 0) dipakai sebagai holdout.** Fold ini ditetapkan sebelum model dilatih, bukan dipilih berdasarkan hasil.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
group_folds = list(sgkf.split(df['review_text'], df['sentiment'], groups=df['review_text']))
train_idx_g, test_idx_g = group_folds[0]

X_train_g = df['review_text'].iloc[train_idx_g]
X_test_g = df['review_text'].iloc[test_idx_g]
y_train_g = df['sentiment'].iloc[train_idx_g]
y_test_g = df['sentiment'].iloc[test_idx_g]
test_ids_group = df['review_id'].iloc[test_idx_g].to_numpy()

overlap_g = set(X_train_g) & set(X_test_g)
assert len(overlap_g) == 0, 'Ada teks yang muncul di train dan test group split'

dist_group = pd.DataFrame({
    'train (baris)': y_train_g.value_counts(),
    'test (baris)': y_test_g.value_counts(),
    'train (teks unik)': df.iloc[train_idx_g].drop_duplicates('review_text')['sentiment'].value_counts(),
    'test (teks unik)': df.iloc[test_idx_g].drop_duplicates('review_text')['sentiment'].value_counts(),
}).loc[LABELS]
dist_group.loc['total'] = dist_group.sum()
print('Distribusi label group split (fold 0):')
print(dist_group)
print('\nTeks yang muncul di train dan test sekaligus:', len(overlap_g))
print('\nTeks unik di test group split:')
print(df.iloc[test_idx_g].groupby(['review_text', 'sentiment']).size().rename('frekuensi').reset_index().to_string(index=False))

**Keterbatasan group split.** Test set group split berisi 40 baris, tetapi hanya **8 teks unik** (4 positif, 4 negatif). Training set hanya berisi 32 teks unik. Artinya:
- Salah memprediksi satu teks bisa membuat beberapa baris sekaligus salah, tergantung frekuensi teks tersebut. Metrik jadi sangat sensitif terhadap satu atau dua kesalahan.
- Hasil satu fold bisa berbeda cukup jauh dari fold lain. Karena itu, untuk model Scikit-learn juga ditampilkan hasil 5-fold group cross-validation sebagai cek stabilitas (Section 6.1). Untuk perbandingan langsung dengan Gemini tetap dipakai fold 0.

Dengan data sekecil ini, hasil group split adalah indikasi kasar, bukan estimasi performa produksi yang presisi.

## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer()

t0 = time.perf_counter()
X_train_vec = vectorizer.fit_transform(X_train)   # fit hanya pada data train
t_vec_fit = time.perf_counter() - t0

t0 = time.perf_counter()
X_test_vec = vectorizer.transform(X_test)         # test hanya di-transform
t_vec_test = time.perf_counter() - t0

print('Ukuran vocabulary (dari train):', len(vectorizer.vocabulary_))
print('Shape X_train_vec:', X_train_vec.shape)
print('Shape X_test_vec :', X_test_vec.shape)

**Alasan TF-IDF.** Ulasan di dataset pendek dan sentimennya banyak ditentukan kata kunci (misalnya *puas*, *rapi*, *cepat* vs *rusak*, *lambat*, *kecewa*). TF-IDF mengubah teks menjadi bobot kata dan memberi bobot lebih rendah pada kata yang muncul di hampir semua ulasan.

Parameter dibiarkan default (lowercase, token minimal 2 karakter, unigram). Stopword removal dan stemming sengaja tidak dipakai. Daftar stopword Bahasa Indonesia umumnya memuat kata seperti *tidak* dan *belum*, padahal kata-kata ini membalik makna sentimen.

Vectorizer di-`fit` hanya pada `X_train`. `X_test` hanya di-`transform`, jadi vocabulary dan bobot IDF tidak dipengaruhi data test.

### 4.2 Training Model

In [ ]:
clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

t0 = time.perf_counter()
clf.fit(X_train_vec, y_train)
t_clf_fit = time.perf_counter() - t0

train_time_classic = t_vec_fit + t_clf_fit
print(f'Waktu training (fit TF-IDF + fit Logistic Regression): {train_time_classic*1000:.2f} ms')
clf

**Alasan Logistic Regression.** Model linear ini umum dipakai sebagai baseline klasifikasi teks dengan fitur TF-IDF: cepat dilatih, tidak butuh GPU, dan bobotnya bisa dibaca untuk melihat kata mana yang mendorong prediksi. Hyperparameter memakai default (`C=1.0`, solver `lbfgs`). `max_iter` dinaikkan agar optimisasi pasti konvergen. **Tidak ada tuning hyperparameter**, supaya tidak ada keputusan model yang diambil berdasarkan test set.

Kata dengan bobot paling kuat untuk masing-masing kelas (bobot positif mendorong ke `positif`):

In [ ]:
coef = pd.Series(clf.coef_[0], index=vectorizer.get_feature_names_out())
print('Kelas yang dianggap positif oleh model:', clf.classes_[1])
top_words = pd.DataFrame({
    'kata → negatif': coef.nsmallest(10).index,
    'bobot neg': coef.nsmallest(10).round(3).values,
    'kata → positif': coef.nlargest(10).index,
    'bobot pos': coef.nlargest(10).round(3).values,
})
top_words

**Catatan.** Kata seperti *lambat*, *rusak*, *komplain* (negatif) dan *cepat*, *rapi*, *puas*, *recommended* (positif) masuk akal sebagai penanda sentimen. Tapi ada juga kata netral dengan bobot besar, misalnya *sudah*, *yang*, *seperti* ke arah negatif dan *ada*, *lebih*, *seller* ke arah positif. Ini terjadi karena dataset hanya berisi 40 kalimat yang diulang-ulang, sehingga model juga menghafal kata pengisi dari kalimat tertentu. Pola ini berisiko salah ketika diterapkan ke ulasan dengan susunan kata berbeda.

### 4.3 Prediksi pada Test Set

In [ ]:
t0 = time.perf_counter()
y_pred_classic = clf.predict(X_test_vec)
t_clf_pred = time.perf_counter() - t0

infer_time_classic = t_vec_test + t_clf_pred   # transform test + predict
print(f'Waktu inference {len(X_test)} review (transform + predict): {infer_time_classic*1000:.2f} ms')
print(f'Rata-rata per review: {infer_time_classic/len(X_test)*1000:.4f} ms')

Satu kali pengukuran di level milidetik mudah terpengaruh noise (cache, proses lain di laptop). Untuk angka yang lebih stabil, training dan inference diulang 20 kali dan diambil **median**-nya. Waktu diukur dengan `time.perf_counter()` di mesin lokal (CPU), mencakup pemrosesan TF-IDF dan model.

In [ ]:
def time_classic_once():
    vec = TfidfVectorizer()
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    t0 = time.perf_counter()
    model.fit(vec.fit_transform(X_train), y_train)
    t1 = time.perf_counter()
    model.predict(vec.transform(X_test))
    t2 = time.perf_counter()
    return t1 - t0, t2 - t1

timings = np.array([time_classic_once() for _ in range(20)])
train_time_classic_median = np.median(timings[:, 0])
infer_time_classic_median = np.median(timings[:, 1])
print(f'Median waktu training  (20x): {train_time_classic_median*1000:.2f} ms')
print(f'Median waktu inference (20x): {infer_time_classic_median*1000:.2f} ms untuk {len(X_test)} review '
      f'({infer_time_classic_median/len(X_test)*1000:.4f} ms/review)')

Pengukuran pertama di cell 4.2 (puluhan ms) lebih lambat daripada median 20 kali ulang (sekitar 2 ms). Pada pemanggilan pertama ada overhead inisialisasi library. Angka median dipakai untuk perbandingan di Section 7. Waktu training dan inference adalah dua aktivitas berbeda dan dilaporkan terpisah.

Hasil prediksi model klasik dibandingkan dengan ground truth:

In [ ]:
hasil_classic = pd.DataFrame({
    'review_id': test_ids_main,
    'review_text': X_test.values,
    'ground_truth': y_test.values,
    'pred_classic': y_pred_classic,
}, index=X_test.index)
hasil_classic['benar'] = hasil_classic['ground_truth'] == hasil_classic['pred_classic']

print(f"Prediksi benar: {hasil_classic['benar'].sum()} dari {len(hasil_classic)}")
print('\nPrediksi yang salah:')
display(hasil_classic.loc[~hasil_classic['benar']])
hasil_classic.head(10)

#### Model klasik pada group split (evaluasi tambahan)

Model dilatih ulang dari nol pada `X_train_g`. Konfigurasinya sama persis dengan model utama (TF-IDF default + Logistic Regression `max_iter=1000`), hanya dibungkus `Pipeline` agar fit vectorizer tetap terbatas pada data train.

In [ ]:
pipe_g = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)
pipe_g.fit(X_train_g, y_train_g)
y_pred_classic_g = pipe_g.predict(X_test_g)

hasil_classic_g = pd.DataFrame({
    'review_id': test_ids_group,
    'review_text': X_test_g.values,
    'ground_truth': y_test_g.values,
    'pred_classic': y_pred_classic_g,
}, index=X_test_g.index)
hasil_classic_g['benar'] = hasil_classic_g['ground_truth'] == hasil_classic_g['pred_classic']
print(f"Prediksi benar (group split): {hasil_classic_g['benar'].sum()} dari {len(hasil_classic_g)}")

# ringkasan per teks unik: frekuensi dan apakah prediksinya benar
hasil_classic_g.groupby(['review_text', 'ground_truth', 'pred_classic']).size().rename('frekuensi').reset_index()

## 5. Pendekatan 2 — LLM API (Gemini)

### 5.1 Setup API

In [ ]:
# Simpan API key di environment variable, JANGAN hardcode langsung di notebook
# API key dibaca dari file .env di root project (file ini tidak di-commit ke GitHub)
load_dotenv(os.path.join('..', '.env'))
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError('GEMINI_API_KEY belum ditemukan. Buat file .env di root project berisi: GEMINI_API_KEY=<api key>')

client = genai.Client(api_key=GEMINI_API_KEY)
print('GEMINI_API_KEY ditemukan, client Gemini siap dipakai (nilai key tidak ditampilkan).')

**Parameter eksperimen Gemini**

| Parameter | Nilai | Alasan |
|---|---|---|
| Model | `gemini-3.5-flash-lite` | Model `gemini-2.0-flash` di starter tidak lagi tersedia untuk API key ini (dicek lewat `models.list`). Varian flash-lite dipilih karena tugasnya klasifikasi teks pendek, latensinya rendah, dan biayanya murah untuk volume ulasan yang besar. `gemini-2.5-flash` tidak dipilih karena di halaman harga resmi sudah ditandai deprecated. |
| `temperature` | `0.0` | Klasifikasi butuh jawaban yang konsisten, bukan variasi kreatif. Temperature 0 membuat model memilih token paling mungkin, sehingga teks yang sama sebaiknya mendapat label yang sama. Konsistensi ini dicek di Section 5.3. |
| `thinking_level` | `minimal` | Gemini 3.x mendukung *thinking*. Untuk klasifikasi kalimat pendek, penalaran panjang tidak diperlukan dan hanya menambah latensi serta token output yang ditagih. `minimal` juga nilai default untuk model ini; ditulis eksplisit agar tercatat. |
| `response_mime_type` + `response_schema` | `text/x.enum`, enum `['negatif', 'positif']` | Output dibatasi ke dua label yang valid, sehingga tidak ada jawaban seperti "Sentimen: Positif." atau penjelasan panjang. Output tetap dinormalisasi dan divalidasi di kode untuk berjaga-jaga. |
| Retry | maks. 6 percobaan, jeda 5, 10, 20, 40, 60 detik | Hanya untuk error sementara: rate limit (429), server error (5xx), atau gangguan jaringan. Error lain (misalnya key tidak valid) langsung dicatat sebagai gagal. |

Harga resmi `gemini-3.5-flash-lite` (paid tier, input teks / output termasuk thinking): **US$0,30 / US$2,50 per 1 juta token**. Sumber: https://ai.google.dev/gemini-api/docs/pricing, yang saat diakses dialihkan ke https://aistudio.google.com/docs/pricing (diakses 15 November 2026). Model ini juga punya free tier.

In [ ]:
MODEL_NAME = 'gemini-3.5-flash-lite'

GEN_CONFIG = types.GenerateContentConfig(
    temperature=0.0,
    thinking_config=types.ThinkingConfig(thinking_level='minimal'),
    response_mime_type='text/x.enum',
    response_schema={'type': 'STRING', 'enum': LABELS},
)

# Harga resmi per 1 juta token (paid tier), dicek 15 November 2026
PRICE_INPUT_PER_1M = 0.30    # US$
PRICE_OUTPUT_PER_1M = 2.50   # US$, output + thinking tokens

model_info = client.models.get(model=MODEL_NAME)
print('Model    :', model_info.name, '| versi:', model_info.version)
print('Config   :', GEN_CONFIG.model_dump(exclude_none=True))

### 5.2 Merancang Prompt

Pendekatan yang dipakai adalah **zero-shot**: model hanya diberi instruksi dan definisi label, tanpa contoh ulasan berlabel. Alasannya:
- Perbandingannya jadi jelas. Gemini sama sekali tidak memakai data training, sehingga terlihat seberapa jauh LLM bisa bekerja tanpa proses training.
- Few-shot butuh contoh. Kalau contohnya diambil dari dataset ini (yang hanya punya 40 kalimat), besar kemungkinan contoh tersebut identik dengan kalimat di test set, dan itu menjadi bentuk leakage baru.
- Prompt dirancang sebelum Gemini dijalankan dan tidak diubah setelah melihat hasil, agar tidak terjadi tuning prompt terhadap test set.

Prompt yang dipakai:

```
Tentukan sentimen ulasan pelanggan e-commerce berikut.

Pilihan label:
- positif: pelanggan menunjukkan rasa puas terhadap produk, pengiriman, atau pelayanan penjual.
- negatif: pelanggan menunjukkan rasa kecewa atau keluhan terhadap produk, pengiriman, atau pelayanan penjual.

Nilai hanya berdasarkan isi ulasan. Jawab dengan satu kata saja: positif atau negatif.

Ulasan: """{review_text}"""
Sentimen:
```

Prompt ditulis dalam Bahasa Indonesia, sama dengan bahasa ulasan. Definisi label disamakan dengan Problem Statement di Section 1. Ulasan dibungkus tanda kutip tiga agar batas teks ulasan jelas bagi model. Instruksi "jawab satu kata" dipertahankan dari starter meskipun output sudah dibatasi enum di config.

In [ ]:
# 🔧 Sesuaikan label pada prompt jika dataset final menggunakan label berbeda
from google.genai import errors as genai_errors

PROMPT_TEMPLATE = '''Tentukan sentimen ulasan pelanggan e-commerce berikut.

Pilihan label:
- positif: pelanggan menunjukkan rasa puas terhadap produk, pengiriman, atau pelayanan penjual.
- negatif: pelanggan menunjukkan rasa kecewa atau keluhan terhadap produk, pengiriman, atau pelayanan penjual.

Nilai hanya berdasarkan isi ulasan. Jawab dengan satu kata saja: positif atau negatif.

Ulasan: """{review_text}"""
Sentimen:'''

RETRY_WAITS = [5, 10, 20, 40, 60]   # detik; maksimal 6 percobaan
RETRYABLE_CODES = {429, 500, 502, 503, 504}


def normalize_label(raw):
    """Ubah output mentah LLM menjadi 'positif'/'negatif', atau 'INVALID' jika tidak bisa dipastikan."""
    if raw is None:
        return 'INVALID'
    text = raw.strip().lower().strip(' .,:;!"\'`*\n')
    if text in LABELS:
        return text
    found = [lab for lab in LABELS if lab in text]
    return found[0] if len(found) == 1 else 'INVALID'


def classify_sentiment_llm(review_text):
    """Kirim satu ulasan ke Gemini. Mengembalikan dict berisi label, output mentah, status, latensi, dan token."""
    prompt = PROMPT_TEMPLATE.format(review_text=review_text)
    result = {'raw_output': None, 'pred_llm': 'GAGAL', 'status': 'gagal', 'error': None,
              'attempts': 0, 'retry_errors': '', 'latency_s': None,
              'prompt_tokens': None, 'output_tokens': None, 'thought_tokens': None, 'total_tokens': None}
    retry_log = []   # error dari percobaan yang kemudian diulang
    for attempt in range(len(RETRY_WAITS) + 1):
        result['attempts'] = attempt + 1
        try:
            t0 = time.perf_counter()
            response = client.models.generate_content(model=MODEL_NAME, contents=prompt, config=GEN_CONFIG)
            result['latency_s'] = time.perf_counter() - t0
        except Exception as e:
            code_ = getattr(e, 'code', None)
            result['error'] = f'{type(e).__name__} {code_}: {str(e)[:200]}'
            retryable = (isinstance(e, genai_errors.APIError) and code_ in RETRYABLE_CODES) or \
                        not isinstance(e, genai_errors.APIError)   # error jaringan juga dicoba ulang
            if retryable and attempt < len(RETRY_WAITS):
                retry_log.append(result['error'])
                result['retry_errors'] = ' || '.join(retry_log)
                time.sleep(RETRY_WAITS[attempt])
                continue
            return result   # tetap 'GAGAL', tidak diubah menjadi label apa pun

        usage = response.usage_metadata
        result.update({
            'raw_output': response.text,
            'error': None,
            'prompt_tokens': usage.prompt_token_count,
            'output_tokens': usage.candidates_token_count,
            'thought_tokens': usage.thoughts_token_count or 0,
            'total_tokens': usage.total_token_count,
        })
        label = normalize_label(response.text)
        result['pred_llm'] = label
        result['status'] = 'ok' if label in LABELS else 'output_tidak_valid'
        return result
    return result


# Contoh isi prompt untuk satu ulasan (tanpa memanggil API)
print(PROMPT_TEMPLATE.format(review_text=X_test.iloc[0]))

### 5.3 Menjalankan Prediksi pada Test Set

Catatan: lakukan normalisasi terhadap output LLM sebelum dibandingkan dengan label asli (misal: lowercase, strip whitespace).

Cara kerja cell di bawah:
- Setiap baris test dikirim **satu per satu**, termasuk teks yang duplikat. Dengan begitu konsistensi jawaban Gemini untuk teks identik bisa dicek, dan jumlah request sama dengan skenario produksi (satu ulasan, satu request).
- Gemini dijalankan pada **test set split utama (40 baris)** dan **test set group split (40 baris)**, yaitu `X_test` dan `X_test_g` yang sama dengan yang dipakai model Scikit-learn.
- Output dinormalisasi oleh `normalize_label` (lowercase, strip spasi dan tanda baca). Output di luar dua label dicatat sebagai `INVALID`. Request yang gagal setelah retry dicatat sebagai `GAGAL`. Keduanya **tidak** diubah menjadi `negatif` atau `positif`.
- Hasil (output mentah, label, latensi, token) disimpan ke `../data/gemini_predictions.csv`. Kalau file ini sudah ada dan isinya cocok dengan test set saat ini, notebook memakai hasil tersimpan agar API tidak dipanggil ulang setiap kali notebook dijalankan, dan agar angka yang dilaporkan tetap sama. Set `RERUN_GEMINI = True` untuk memanggil API lagi.
- Waktu yang dicatat: (1) latensi tiap request yang berhasil, yaitu durasi panggilan `generate_content` saja, dan (2) total waktu loop, yang juga mencakup jeda retry kalau ada.

In [ ]:
GEMINI_CACHE = '../data/gemini_predictions.csv'
RERUN_GEMINI = False


def run_gemini(X, y, ids, split_name):
    rows = []
    t_start = time.perf_counter()
    for review_id, review, label in zip(ids, X, y):
        r = classify_sentiment_llm(review)
        rows.append({'split': split_name, 'review_id': review_id, 'review_text': review,
                     'ground_truth': label, **r})
        if r['status'] != 'ok':
            print(f"  [{split_name}] review_id={review_id}: {r['status']} | {r['error'] or r['raw_output']}")
    wall = time.perf_counter() - t_start
    out = pd.DataFrame(rows)
    out['loop_wall_time_s'] = wall
    print(f'{split_name}: {len(out)} request selesai dalam {wall:.1f} detik')
    return out


def cache_matches(cached):
    expected = {'utama': list(test_ids_main), 'group': list(test_ids_group)}
    return all(cached.loc[cached['split'] == s, 'review_id'].tolist() == ids for s, ids in expected.items())


cached = pd.read_csv(GEMINI_CACHE) if os.path.exists(GEMINI_CACHE) else None
if cached is not None and not RERUN_GEMINI and cache_matches(cached):
    gemini_results = cached
    print(f'Memakai hasil Gemini tersimpan dari {GEMINI_CACHE} (dijalankan: {cached["run_timestamp"].iloc[0]}).')
else:
    run_ts = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    res_main = run_gemini(X_test, y_test, test_ids_main, 'utama')
    res_group = run_gemini(X_test_g, y_test_g, test_ids_group, 'group')
    gemini_results = pd.concat([res_main, res_group], ignore_index=True)
    gemini_results['model'] = MODEL_NAME
    gemini_results['run_timestamp'] = run_ts
    gemini_results.to_csv(GEMINI_CACHE, index=False)
    print(f'Hasil disimpan ke {GEMINI_CACHE}')

res_main = gemini_results[gemini_results['split'] == 'utama'].reset_index(drop=True)
res_group = gemini_results[gemini_results['split'] == 'group'].reset_index(drop=True)

# Urutan harus sama persis dengan test set Scikit-learn
assert res_main['review_id'].tolist() == list(test_ids_main)
assert res_group['review_id'].tolist() == list(test_ids_group)

# TODO dari starter (sudah dikerjakan): y_pred_llm dinormalisasi ke 'positif'/'negatif' oleh normalize_label()
y_pred_llm = res_main['pred_llm'].tolist()
y_pred_llm_g = res_group['pred_llm'].tolist()

Ringkasan status request, latensi, dan penggunaan token:

In [ ]:
status_summary = gemini_results.groupby('split')['status'].value_counts().unstack(fill_value=0)
for col in ['ok', 'output_tidak_valid', 'gagal']:
    if col not in status_summary:
        status_summary[col] = 0
status_summary = status_summary[['ok', 'output_tidak_valid', 'gagal']]
status_summary['total'] = status_summary.sum(axis=1)
print('Status request per split:')
display(status_summary)
print('Jumlah percobaan per request:', gemini_results['attempts'].value_counts().sort_index().to_dict())
if 'retry_errors' in gemini_results:
    retried = gemini_results[gemini_results['attempts'] > 1]
    print('Error yang memicu retry:')
    print(retried[['split', 'review_id', 'attempts', 'retry_errors']].to_string(index=False))
else:
    print('Catatan: hasil tersimpan berasal dari run sebelum kolom retry_errors ditambahkan, '
          'sehingga penyebab retry (kemungkinan rate limit) tidak tercatat.')

ok = gemini_results[gemini_results['status'] == 'ok']
lat = ok['latency_s']
print(f'\nLatensi per request berhasil (n={len(ok)}): median {lat.median():.3f} s | '
      f'rata-rata {lat.mean():.3f} s | min {lat.min():.3f} s | maks {lat.max():.3f} s')
for s, g in gemini_results.groupby('split'):
    print(f"Total waktu loop split {s:5s}: {g['loop_wall_time_s'].iloc[0]:.1f} s untuk {len(g)} review")

tok = gemini_results[['prompt_tokens', 'output_tokens', 'thought_tokens', 'total_tokens']].sum()
print('\nTotal token seluruh request:')
print(tok.astype(int).to_string())
cost_actual = (tok['prompt_tokens'] * PRICE_INPUT_PER_1M +
               (tok['output_tokens'] + tok['thought_tokens']) * PRICE_OUTPUT_PER_1M) / 1e6
print(f'\nEstimasi biaya eksperimen ini jika memakai tarif paid tier: US${cost_actual:.6f}')

Contoh hasil prediksi Gemini dibandingkan dengan ground truth (split utama):

In [ ]:
cols = ['review_id', 'review_text', 'ground_truth', 'raw_output', 'pred_llm', 'status', 'latency_s', 'total_tokens']
display(res_main[cols].head(10))
print('Baris split utama dengan prediksi berbeda dari ground truth:')
res_main.loc[res_main['pred_llm'] != res_main['ground_truth'], cols]

Cek konsistensi: apakah teks yang identik selalu mendapat label yang sama dari Gemini (gabungan kedua split)?

In [ ]:
consistency = (gemini_results[gemini_results['status'] == 'ok']
               .groupby('review_text')['pred_llm']
               .agg(jumlah_request='size', label_berbeda='nunique', label=lambda s: sorted(set(s))))
dup_texts = consistency[consistency['jumlah_request'] > 1]
print('Teks unik yang dikirim lebih dari sekali:', len(dup_texts))
print('Teks dengan jawaban tidak konsisten     :', (dup_texts['label_berbeda'] > 1).sum())
dup_texts[dup_texts['label_berbeda'] > 1]

Hasil Gemini pada group split (ringkasan per teks unik):

In [ ]:
res_group.groupby(['review_text', 'ground_truth', 'pred_llm']).size().rename('frekuensi').reset_index()

## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik

**Definisi metrik yang dipakai untuk kedua model**
- **Accuracy**: proporsi prediksi yang benar dari seluruh sampel.
- **Precision / Recall / F1 (binary, `pos_label='positif'`)**: metrik utama. Kelas `positif` dianggap kelas target.
  - Precision: dari semua yang diprediksi positif, berapa yang benar positif.
  - Recall: dari semua ulasan yang benar positif, berapa yang berhasil dikenali.
  - F1: rata-rata harmonik precision dan recall.
- **Precision / Recall / F1 macro-average**: rata-rata sederhana metrik kelas `positif` dan `negatif`. Ditampilkan sebagai pembanding karena kelas `negatif` sama pentingnya bagi tim produk (keluhan pelanggan), dan proporsi kelas tidak persis 50:50.
- **Confusion matrix**: baris = label asli, kolom = prediksi, urutan `negatif`, `positif`.
  - *False Positive*: ulasan negatif diprediksi positif. Keluhan pelanggan jadi tidak terlihat.
  - *False Negative*: ulasan positif diprediksi negatif. Produk terlihat lebih buruk dari kenyataannya.

Fungsi helper di bawah dipakai untuk semua evaluasi (model klasik, Gemini, split utama, dan group split) agar definisinya identik.

In [ ]:
def compute_metrics(y_true, y_pred, name):
    y_true, y_pred = list(y_true), list(y_pred)
    p_b, r_b, f_b, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[POS_LABEL], average='binary', pos_label=POS_LABEL, zero_division=0)
    p_m, r_m, f_m, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, average='macro', zero_division=0)
    return {
        'Model / evaluasi': name,
        'n sampel': len(y_true),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision (positif)': p_b, 'Recall (positif)': r_b, 'F1 (positif)': f_b,
        'Precision (macro)': p_m, 'Recall (macro)': r_m, 'F1 (macro)': f_m,
    }


def plot_cm(y_true, y_pred, title, filename=None):
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    fig, ax = plt.subplots(figsize=(4.6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=LABELS, yticklabels=LABELS, annot_kws={'size': 14}, ax=ax)
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Label asli (ground truth)')
    ax.set_title(title, fontsize=11)
    plt.tight_layout()
    if filename:
        fig.savefig(os.path.join(DOC_DIR, filename), dpi=150, bbox_inches='tight')
    plt.show()
    tn, fp, fn, tp = cm.ravel()
    print(f'TN (negatif benar)={tn} | FP (negatif → positif)={fp} | '
          f'FN (positif → negatif)={fn} | TP (positif benar)={tp}')
    return cm


def show_metrics(rows):
    return pd.DataFrame(rows).set_index('Model / evaluasi').round(4)

In [ ]:
print('=== Model Klasik (Scikit-learn) — split utama ===')
print('Accuracy:', accuracy_score(y_test, y_pred_classic))
print(classification_report(y_test, y_pred_classic, labels=LABELS, digits=4, zero_division=0))
print('Confusion Matrix (baris = asli, kolom = prediksi, urutan negatif, positif):\n',
      confusion_matrix(y_test, y_pred_classic, labels=LABELS))

In [ ]:
metrics_classic = compute_metrics(y_test, y_pred_classic, 'Scikit-learn — split utama')
show_metrics([metrics_classic])

In [ ]:
cm_classic = plot_cm(y_test, y_pred_classic,
                     'Scikit-learn (TF-IDF + LogReg)\nSplit utama, n=40',
                     'confusion_matrix_sklearn.png')

**Interpretasi: Scikit-learn pada split utama**
- **Accuracy = 1,0000**: ke-40 review test diprediksi benar.
- **Precision (positif) = 1,0000**: semua review yang diprediksi `positif` memang positif.
- **Recall (positif) = 1,0000**: ke-22 review positif dikenali semua.
- **F1 (positif) = 1,0000** dan **F1 macro = 1,0000**: tidak ada kesalahan di kelas mana pun.
- **Confusion matrix**: TN=18, TP=22, FP=0, FN=0. Tidak ada keluhan yang terlewat (FP) dan tidak ada ulasan positif yang dicap negatif (FN).

Angka sempurna ini **tidak berarti model sudah siap untuk review baru**. Section 3 menunjukkan 39 dari 40 review test punya kembaran persis di training set. Model pada dasarnya diuji pada kalimat yang sudah ia lihat bersama labelnya. Skor ini lebih tepat dibaca sebagai bukti bahwa model berhasil mempelajari 40 kalimat di dataset, bukan sebagai kemampuan generalisasi.

#### Analisis tambahan 1: 26 teks unik di test set utama

Setiap teks unik dihitung satu kali, sehingga teks yang sering muncul tidak mendominasi skor. **Teks-teks ini tetap ada di training set**, jadi analisis ini bukan evaluasi bebas leakage. Tujuannya hanya melihat apakah frekuensi duplikasi mengubah gambaran metrik.

In [ ]:
uniq_classic = hasil_classic.drop_duplicates('review_text')
metrics_classic_unique = compute_metrics(uniq_classic['ground_truth'], uniq_classic['pred_classic'],
                                         'Scikit-learn — 26 teks unik (split utama)')
print('Distribusi label teks unik:', uniq_classic['ground_truth'].value_counts().to_dict())
show_metrics([metrics_classic, metrics_classic_unique])

**Interpretasi.** Ketika setiap teks dihitung satu kali (26 teks, 13 positif dan 13 negatif), semua metrik tetap 1,0000. Jadi skor sempurna di split utama bukan karena beberapa teks yang sering muncul kebetulan benar. Hampir semua teks memang sudah dihafal model. Satu-satunya teks test yang tidak ada di training set (*"Tidak sesuai standar, banyak cacat pada produk yang diterima."*) juga diprediksi benar, tapi satu sampel tidak cukup untuk menarik kesimpulan.

#### Analisis tambahan 2: group split (teks test tidak pernah dilihat saat training)

In [ ]:
metrics_classic_g = compute_metrics(y_test_g, y_pred_classic_g, 'Scikit-learn — group split (fold 0)')
show_metrics([metrics_classic, metrics_classic_g])

In [ ]:
cm_classic_g = plot_cm(y_test_g, y_pred_classic_g,
                       'Scikit-learn (TF-IDF + LogReg)\nGroup split fold 0, n=40',
                       'confusion_matrix_sklearn_group.png')

**Interpretasi: Scikit-learn pada group split (fold 0)**

Di evaluasi ini, teks test benar-benar belum pernah dilihat model. Hasilnya turun jauh:
- **Accuracy = 0,6250** (25 dari 40 baris benar).
- **Precision (positif) = 0,6842**: dari 19 review yang diprediksi positif, 6 sebenarnya negatif.
- **Recall (positif) = 0,5909**: hanya 13 dari 22 review positif yang dikenali.
- **F1 (positif) = 0,6341**, **F1 macro = 0,6248**: performa di kedua kelas sama-sama lemah.
- **Confusion matrix**: TN=12, FP=6, FN=9, TP=13.

Kalau dihitung per teks unik, model salah di **2 dari 8 teks**. Karena kedua teks itu sering muncul, kesalahannya menjadi 15 baris:
1. *"Produknya rapi dan bersih, tidak ada cacat sama sekali."* (positif, muncul 9 kali) diprediksi **negatif** (False Negative). Kata *produknya* dan *bersih* tidak ada di vocabulary training. Yang tersisa adalah *tidak*, *cacat*, dan *sekali*, yang di training set lebih sering muncul di ulasan negatif. Model berbasis kata tidak memahami bahwa *"tidak ada cacat"* justru pujian.
2. *"Bahan murahan, tidak sesuai dengan harga yang dibayarkan."* (negatif, muncul 6 kali) diprediksi **positif** (False Positive). Kata kunci *murahan* tidak pernah muncul di training set (out-of-vocabulary), sedangkan *bahan*, *sesuai*, dan *dengan* berbobot positif karena berasal dari ulasan positif seperti *"Bahan nyaman dipakai…"* dan *"…sesuai deskripsi"*.

Dalam konteks e-commerce, FP berarti keluhan soal kualitas bahan tidak terdeteksi. FN berarti ulasan pujian ikut masuk daftar ulasan negatif. Keduanya menunjukkan keterbatasan utama TF-IDF + Logistic Regression: model ini tidak menangkap negasi dan tidak bisa menilai kata yang tidak ada di data training.

Cek stabilitas model klasik di kelima fold group split. Pengecekan ini **hanya untuk Scikit-learn** dan tidak dipakai untuk perbandingan langsung dengan Gemini. Konfigurasi model tetap sama; tidak ada fold atau parameter yang dipilih dari hasil ini.

In [ ]:
cv_rows = []
for k, (tr, te) in enumerate(group_folds):
    model_k = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    model_k.fit(df['review_text'].iloc[tr], df['sentiment'].iloc[tr])
    pred_k = model_k.predict(df['review_text'].iloc[te])
    row = compute_metrics(df['sentiment'].iloc[te], pred_k, f'fold {k}')
    row['teks unik test'] = df['review_text'].iloc[te].nunique()
    cv_rows.append(row)

cv_classic = show_metrics(cv_rows)
display(cv_classic[['n sampel', 'teks unik test', 'Accuracy', 'F1 (positif)', 'F1 (macro)']])
print('Rata-rata ± std 5 fold:')
cv_classic[['Accuracy', 'F1 (positif)', 'F1 (macro)']].agg(['mean', 'std']).round(4)

**Interpretasi cek stabilitas.** Di kelima fold group split, accuracy model klasik berkisar dari 0,38 sampai 0,80, dengan rata-rata 0,64 ± 0,16. Fold 0 (0,625) tidak dipilih karena paling buruk atau paling baik; posisinya dekat rata-rata. Variasi yang besar ini wajar karena setiap fold hanya berisi 7–9 teks unik, sehingga satu teks yang salah bisa menggeser accuracy hingga belasan persen. Kesimpulan yang aman: **pada teks yang belum pernah dilihat, model klasik dengan data 40 kalimat ini kira-kira berada di kisaran 60–70% accuracy, dengan ketidakpastian tinggi.** Angka 100% di split utama tidak mewakili performa tersebut.

### 6.2 Evaluasi LLM API

Sebelum menghitung metrik, cek dulu apakah ada prediksi Gemini yang `GAGAL` atau `INVALID`. Kalau ada, metrik **kedua model** dihitung ulang pada subset sampel yang sama-sama valid, dan jumlah sampelnya ditulis di tabel. Prediksi yang gagal tidak diganti dengan label apa pun.

In [ ]:
valid_main = np.array([p in LABELS for p in y_pred_llm])
valid_group = np.array([p in LABELS for p in y_pred_llm_g])
print(f'Prediksi Gemini valid — split utama: {valid_main.sum()}/{len(valid_main)} | group split: {valid_group.sum()}/{len(valid_group)}')

# Subset evaluasi yang sama untuk kedua model (saat ini = seluruh test set karena semua valid)
y_true_main_eval = y_test.to_numpy()[valid_main]
pred_classic_main_eval = np.asarray(y_pred_classic)[valid_main]
pred_llm_main_eval = np.asarray(y_pred_llm)[valid_main]
y_true_group_eval = y_test_g.to_numpy()[valid_group]
pred_classic_group_eval = np.asarray(y_pred_classic_g)[valid_group]
pred_llm_group_eval = np.asarray(y_pred_llm_g)[valid_group]

if valid_main.all() and valid_group.all():
    print('Semua prediksi valid, jadi kedua model dievaluasi pada test set yang identik (40 + 40 baris).')
else:
    print('PERINGATAN: ada prediksi Gemini yang tidak valid. Metrik kedua model dihitung pada subset valid yang sama.')

In [ ]:
print('=== LLM API (Gemini) — split utama ===')
print('Accuracy:', accuracy_score(y_test, y_pred_llm))
print(classification_report(y_test, y_pred_llm, labels=LABELS, digits=4, zero_division=0))
print('Confusion Matrix (baris = asli, kolom = prediksi, urutan negatif, positif):\n',
      confusion_matrix(y_test, y_pred_llm, labels=LABELS))

In [ ]:
metrics_llm = compute_metrics(y_true_main_eval, pred_llm_main_eval, 'Gemini — split utama')
show_metrics([metrics_llm])

In [ ]:
cm_llm = plot_cm(y_true_main_eval, pred_llm_main_eval,
                 f'Gemini ({MODEL_NAME}, zero-shot)\nSplit utama, n={len(y_true_main_eval)}',
                 'confusion_matrix_gemini.png')

**Interpretasi: Gemini pada split utama**
- **Accuracy = 1,0000**: ke-40 review test diklasifikasikan benar.
- **Precision (positif) = 1,0000**: tidak ada review negatif yang dilabeli positif.
- **Recall (positif) = 1,0000**: ke-22 review positif dikenali semua.
- **F1 (positif) = 1,0000**, **F1 macro = 1,0000**: performa sama baiknya di kelas positif dan negatif.
- **Confusion matrix**: TN=18, TP=22, FP=0, FN=0. Tidak ada keluhan yang terlewat dan tidak ada pujian yang dicap negatif.

Berbeda dengan Scikit-learn, skor ini **tidak dipengaruhi** duplikasi train/test, karena Gemini tidak dilatih dengan data training sama sekali. Meski begitu, 40 baris ini hanya berisi 26 kalimat unik yang pendek dan sentimennya eksplisit, jadi skor sempurna di sini belum menjamin hasil yang sama untuk ulasan nyata yang lebih beragam (lihat Section 7).

#### Analisis tambahan 1: 26 teks unik di test set utama

In [ ]:
uniq_llm = res_main.drop_duplicates('review_text')   # konsistensi per teks sudah dicek di 5.3
metrics_llm_unique = compute_metrics(uniq_llm['ground_truth'], uniq_llm['pred_llm'], 'Gemini — 26 teks unik (split utama)')
show_metrics([metrics_classic_unique, metrics_llm_unique])

**Interpretasi.** Dengan setiap teks dihitung satu kali, Gemini tetap benar di 26 dari 26 teks, sama seperti Scikit-learn. Di split utama tidak ada perbedaan antara kedua model. Pembeda yang sebenarnya baru terlihat di group split, di mana model klasik tidak lagi bisa mengandalkan kalimat yang sudah dihafal.

#### Analisis tambahan 2: group split (test set yang sama dengan model klasik)

In [ ]:
metrics_llm_g = compute_metrics(y_true_group_eval, pred_llm_group_eval, 'Gemini — group split (fold 0)')
show_metrics([metrics_classic_g, metrics_llm_g])

In [ ]:
cm_llm_g = plot_cm(y_true_group_eval, pred_llm_group_eval,
                   f'Gemini ({MODEL_NAME}, zero-shot)\nGroup split fold 0, n={len(y_true_group_eval)}',
                   'confusion_matrix_gemini_group.png')

**Interpretasi: Gemini pada group split**

Pada 40 baris test group split (8 teks unik yang tidak ada di training set model klasik), Gemini mendapat **Accuracy, Precision, Recall, dan F1 = 1,0000** dengan confusion matrix TN=18, TP=22, FP=0, FN=0. Kedua kalimat yang salah diprediksi Scikit-learn diklasifikasikan benar:
- *"Produknya rapi dan bersih, tidak ada cacat sama sekali."* → `positif` (Gemini memahami bahwa *"tidak ada cacat"* adalah pujian).
- *"Bahan murahan, tidak sesuai dengan harga yang dibayarkan."* → `negatif` (kata *murahan* tidak perlu pernah muncul di data training).

Perlu dicatat, ini 8 kalimat, bukan 40 observasi independen. Hasil 8/8 benar konsisten dengan performa tinggi, tetapi belum cukup untuk menyimpulkan bahwa Gemini akan selalu benar.

### 6.3 Tabel Perbandingan Ringkasan

Semua baris di tabel memakai definisi metrik yang sama (`compute_metrics`). Kolom **Accuracy 95% CI** adalah interval Wilson untuk proporsi prediksi benar. Interval ini dicantumkan untuk menunjukkan seberapa lebar ketidakpastian akibat ukuran test set yang kecil. Karena baris test banyak yang duplikat, interval per baris sebenarnya terlalu optimis; interval per teks unik lebih jujur.

In [ ]:
def wilson_ci(y_true, y_pred, z=1.96):
    n = len(y_true)
    k = int((np.asarray(y_true) == np.asarray(y_pred)).sum())
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return f'[{center - half:.3f}, {center + half:.3f}]'


uniq_classic_g = hasil_classic_g.drop_duplicates('review_text')
uniq_llm_g = res_group.drop_duplicates('review_text')
metrics_classic_g_unique = compute_metrics(uniq_classic_g['ground_truth'], uniq_classic_g['pred_classic'],
                                           'Scikit-learn — group split, 8 teks unik')
metrics_llm_g_unique = compute_metrics(uniq_llm_g['ground_truth'], uniq_llm_g['pred_llm'],
                                       'Gemini — group split, 8 teks unik')

comparison_rows = [
    (compute_metrics(y_true_main_eval, pred_classic_main_eval, 'Scikit-learn — split utama'), y_true_main_eval, pred_classic_main_eval),
    (metrics_llm, y_true_main_eval, pred_llm_main_eval),
    (metrics_classic_unique, uniq_classic['ground_truth'], uniq_classic['pred_classic']),
    (metrics_llm_unique, uniq_llm['ground_truth'], uniq_llm['pred_llm']),
    (compute_metrics(y_true_group_eval, pred_classic_group_eval, 'Scikit-learn — group split (fold 0)'), y_true_group_eval, pred_classic_group_eval),
    (metrics_llm_g, y_true_group_eval, pred_llm_group_eval),
    (metrics_classic_g_unique, uniq_classic_g['ground_truth'], uniq_classic_g['pred_classic']),
    (metrics_llm_g_unique, uniq_llm_g['ground_truth'], uniq_llm_g['pred_llm']),
]
comparison = show_metrics([m for m, _, _ in comparison_rows])
comparison.insert(2, 'Accuracy 95% CI', [wilson_ci(t, p) for _, t, p in comparison_rows])
comparison.insert(1, 'Leakage teks train→test', ['ya (39/40)', 'tidak relevan (tanpa training)',
                                                  'ya (25/26)', 'tidak relevan (tanpa training)',
                                                  'tidak', 'tidak relevan (tanpa training)',
                                                  'tidak', 'tidak relevan (tanpa training)'])
comparison.to_csv('../documentation/comparison_metrics.csv')
comparison

Ringkasan efisiensi kedua pendekatan (angka dari pengukuran di Section 4.3 dan 5.3):

In [ ]:
ok_req = gemini_results[gemini_results['status'] == 'ok']
lat_med = ok_req['latency_s'].median()
tok_sum = gemini_results[['prompt_tokens', 'output_tokens', 'thought_tokens']].sum()

efficiency = pd.DataFrame({
    'Scikit-learn (TF-IDF + LogReg)': {
        'Butuh data berlabel untuk training': 'ya (160 baris)',
        'Waktu training (median 20x)': f'{train_time_classic_median*1000:.2f} ms',
        'Waktu inference per review': f'{infer_time_classic_median/len(X_test)*1000:.4f} ms',
        'Waktu inference 40 review': f'{infer_time_classic_median*1000:.2f} ms',
        'Berjalan di': 'CPU lokal, offline',
        'Token terpakai di eksperimen': '-',
        'Biaya per request': 'tidak ada (biaya hardware lokal)',
    },
    f'Gemini ({MODEL_NAME})': {
        'Butuh data berlabel untuk training': 'tidak (hanya untuk evaluasi)',
        'Waktu training (median 20x)': 'tidak ada training',
        'Waktu inference per review': f'{lat_med:.3f} s (median latensi API)',
        'Waktu inference 40 review': ' / '.join(f"{s}: {g['loop_wall_time_s'].iloc[0]:.1f} s"
                                                 for s, g in gemini_results.groupby('split')) + ' (termasuk retry)',
        'Berjalan di': 'server Google, butuh internet',
        'Token terpakai di eksperimen': f"{int(tok_sum.sum()):,} (80 request)",
        'Biaya per request': f'~US${cost_actual/len(gemini_results):.7f} (tarif paid tier)',
    },
})
efficiency

In [ ]:
# Grafik perbandingan: dua panel (split utama vs group split), tiga metrik, dua model
COLOR_SKLEARN, COLOR_GEMINI = '#2a78d6', '#eb6834'   # slot kategorikal 1 dan 2
metric_cols = ['Accuracy', 'F1 (positif)', 'F1 (macro)']
panels = [('Split utama (n=40)\nScikit-learn sudah melihat 39/40 teks saat training',
           comparison.loc['Scikit-learn — split utama'], comparison.loc['Gemini — split utama']),
          ('Group split fold 0 (n=40)\nteks test tidak ada di training set',
           comparison.loc['Scikit-learn — group split (fold 0)'], comparison.loc['Gemini — group split (fold 0)'])]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
x = np.arange(len(metric_cols))
w = 0.36
for ax, (title, row_sk, row_llm) in zip(axes, panels):
    b1 = ax.bar(x - w/2, row_sk[metric_cols].astype(float), w, color=COLOR_SKLEARN,
                edgecolor='white', linewidth=2, label='Scikit-learn (TF-IDF + LogReg)')
    b2 = ax.bar(x + w/2, row_llm[metric_cols].astype(float), w, color=COLOR_GEMINI,
                edgecolor='white', linewidth=2, label=f'Gemini ({MODEL_NAME})')
    for bars in (b1, b2):
        ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9, color='#333333')
    ax.set_xticks(x, metric_cols)
    ax.set_title(title, fontsize=10.5)
    ax.set_ylim(0, 1.12)
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
axes[0].set_ylabel('Skor (0–1)')
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.suptitle('Perbandingan performa Scikit-learn vs Gemini', y=1.09, fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(DOC_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

**Ringkasan perbandingan**
- **Split utama:** kedua model sama-sama 1,0000 di semua metrik. Bagi Scikit-learn, angka ini hasil dari teks test yang sudah ada di training set (39/40). Bagi Gemini, tidak ada data training yang terlibat.
- **26 teks unik:** keduanya tetap 1,0000. Frekuensi duplikat tidak mengubah kesimpulan.
- **Group split:** perbedaannya jelas. Gemini 1,0000, sedangkan Scikit-learn turun ke Accuracy 0,6250, F1 (positif) 0,6341, F1 macro 0,6248. Per teks unik, Gemini benar 8/8 dan Scikit-learn 6/8. Cek stabilitas 5-fold di Section 6.1 menunjukkan rata-rata accuracy model klasik 0,6433 ± 0,1586, jadi fold 0 bukan kasus yang kebetulan buruk.
- **Ketidakpastian:** interval Wilson per teks unik untuk Gemini di group split masih lebar (8/8 → sekitar [0,68; 1,00]). Keunggulan Gemini di data ini jelas, tetapi besar performanya pada ulasan nyata belum bisa dipastikan dari 8 kalimat.

## 7. Analisis Trade-off dan Limitation

### 7.1 Performa

| Evaluasi | Scikit-learn | Gemini | Catatan |
|---|---|---|---|
| Split utama (40 baris) | Acc 1,000 / F1 1,000 | Acc 1,000 / F1 1,000 | Scikit-learn sudah melihat 39 dari 40 teks test |
| Group split (40 baris, 8 teks baru) | Acc 0,625 / F1 (pos) 0,634 / F1 macro 0,625 | Acc 1,000 / F1 1,000 | Pembanding yang adil untuk teks baru |
| Group 5-fold (hanya Scikit-learn) | Acc rata-rata 0,643 ± 0,159 | — | Variasi tinggi karena tiap fold hanya 7–9 teks |

- Kalau hanya melihat split utama, kedua model tampak setara. Kesimpulan ini menyesatkan karena skor model klasik ditopang oleh duplikasi.
- Pada teks yang belum pernah dilihat, **Gemini lebih unggul**. Kesalahan Scikit-learn punya pola yang bisa dijelaskan: (1) **negasi**, *"tidak ada cacat"* dibaca negatif karena kata *tidak* dan *cacat* berbobot negatif; (2) **kata di luar vocabulary**, *murahan* tidak pernah muncul di training set, sehingga keputusan diambil dari kata netral seperti *bahan* dan *sesuai*. Jenis kesalahan model klasik di group split: 6 False Positive (keluhan tidak terdeteksi) dan 9 False Negative (pujian dicap negatif).
- Di semua test set, Gemini tidak membuat kesalahan. Karena itu pola kesalahan Gemini **tidak bisa dianalisis dari eksperimen ini**, dan itu sendiri termasuk keterbatasan.

### 7.2 Effort implementasi

| Aspek | Scikit-learn | Gemini API |
|---|---|---|
| Data berlabel | Wajib untuk training. Kualitas model sangat bergantung pada variasi data. Dengan 40 kalimat unik, hasilnya tidak cukup untuk generalisasi | Tidak perlu untuk training. Data berlabel tetap dibutuhkan untuk evaluasi |
| Pengerjaan inti | TF-IDF + LogReg cukup beberapa baris kode, bisa dicoba ulang dalam hitungan milidetik | Menulis prompt, mengatur config, mengelola API key, retry/backoff, validasi output, dan caching hasil |
| Pemeliharaan | Retrain jika muncul kosakata atau topik keluhan baru, memantau drift, menyimpan versi model | Memantau perubahan versi model. Di eksperimen ini, model di starter (`gemini-2.0-flash`) sudah tidak tersedia dan `gemini-2.5-flash` sudah deprecated, jadi migrasi model dan evaluasi ulang adalah pekerjaan rutin |

Secara jumlah kode, model klasik lebih sederhana. Secara keseluruhan, effort terbesar model klasik ada di **pengumpulan dan pelabelan data** agar model benar-benar memahami bahasa ulasan. Effort terbesar Gemini ada di **integrasi layanan eksternal** (error handling, rate limit, versi model).

### 7.3 Kecepatan

Cara pengukuran:
- Scikit-learn diukur dengan `time.perf_counter()` di CPU lokal, median dari 20 kali ulang. Training (fit TF-IDF + LogReg) dan inference (transform + predict) dilaporkan **terpisah** karena merupakan aktivitas berbeda.
- Gemini diukur per request (durasi `generate_content`, termasuk jaringan). Total waktu loop dicatat terpisah karena mencakup jeda retry.

| | Scikit-learn | Gemini |
|---|---|---|
| Training | ~2 ms (sekali di awal, diulang saat retrain) | tidak ada |
| Inference per review | ~0,008 ms | median ~0,685 s |
| 40 review | ~0,33 ms | 83,7 s (split utama) dan 166,4 s (group split), sebagian besar karena jeda retry |

Per review, Gemini sekitar **lima orde besaran lebih lambat** daripada model lokal. Untuk fitur label sentimen di halaman produk, latensi ~0,7 detik masih bisa diterima **jika klasifikasi dilakukan secara asinkron** saat ulasan masuk dan hasilnya disimpan, bukan dihitung ulang setiap kali halaman dibuka. Di eksperimen ini 9 dari 80 request perlu dicoba ulang (2–4 percobaan). Artinya, throughput Gemini juga dibatasi kuota dan rate limit, bukan hanya latensi.

### 7.4 Biaya

Biaya aktual eksperimen dihitung dari token yang dilaporkan API (`usage_metadata`) dikali harga resmi paid tier `gemini-3.5-flash-lite` (US$0,30 per 1 juta token input, US$2,50 per 1 juta token output + thinking; dicek 15 November 2026). Tier akun (free atau paid) tidak dicek dari notebook, jadi angka ini adalah **biaya jika memakai tarif paid tier**, bukan tagihan yang pasti. Proyeksi di bawah adalah **estimasi** dengan asumsi panjang ulasan dan prompt sama seperti di eksperimen.

In [ ]:
avg_in = ok_req['prompt_tokens'].mean()
avg_out = (ok_req['output_tokens'] + ok_req['thought_tokens']).mean()
cost_per_review = (avg_in * PRICE_INPUT_PER_1M + avg_out * PRICE_OUTPUT_PER_1M) / 1e6
print(f'Rata-rata token per request: input {avg_in:.1f}, output+thinking {avg_out:.1f}')
print(f'Biaya aktual 80 request (tarif paid tier): US${cost_actual:.6f}')
print(f'Biaya per review: US${cost_per_review:.8f}\n')

projection = pd.DataFrame({'jumlah review': [1_000, 10_000, 100_000, 1_000_000]})
projection['estimasi biaya Gemini (US$)'] = (projection['jumlah review'] * cost_per_review).round(4)
projection['waktu Gemini jika sekuensial (jam)'] = (projection['jumlah review'] * lat_med / 3600).round(2)
projection['waktu inference Scikit-learn (detik)'] = (projection['jumlah review'] * infer_time_classic_median / len(X_test)).round(3)
projection

- Biaya Gemini sangat kecil untuk eksperimen ini (8.305 token, sekitar **US$0,0028** dengan tarif paid tier). Untuk 1 juta ulasan, estimasinya sekitar **US$35,5**. Biaya ini naik linear terhadap jumlah ulasan dan bisa berubah kalau Google mengubah harga; halaman harga yang sama sudah mencantumkan kenaikan harga terjadwal untuk beberapa model lain.
- Scikit-learn tidak punya biaya per request. Biayanya ada di tempat lain: pelabelan data (biasanya yang terbesar), waktu engineer untuk retrain dan evaluasi, dan server kecil untuk menjalankan model. Komponen-komponen ini tidak diukur di eksperimen ini.
- Waktu sekuensial Gemini untuk 1 juta ulasan sekitar 190 jam. Dalam praktik bisa dipercepat dengan request paralel, tetapi tetap dibatasi rate limit akun.

### 7.5 Limitations

**Data dan desain eksperimen**
- Dataset hanya punya 40 kalimat unik (20 per kelas), pendek (6–11 kata), dan sentimennya eksplisit. Tidak ada sarkasme, ulasan campuran, slang berat, typo, atau ulasan panjang. Performa kedua model pada ulasan nyata kemungkinan lebih rendah daripada yang terlihat di sini.
- Split utama mengandung leakage untuk model klasik (39/40 teks test ada di training set). Group split menghilangkan leakage, tetapi test set-nya hanya berisi 8 kalimat, sehingga intervalnya lebar.
- Label dataset dianggap benar dan tidak diverifikasi ulang.
- Hanya ada dua kelas. Ulasan netral atau campuran dipaksa masuk ke positif atau negatif oleh kedua model.

**Scikit-learn (TF-IDF + Logistic Regression)**
- Model cenderung menghafal kalimat. Bobot besar jatuh ke kata netral (*sudah*, *yang*, *ada*, *seller*) karena kalimatnya berulang.
- Tidak memahami negasi atau urutan kata (contoh nyata: *"tidak ada cacat sama sekali"* → salah jadi negatif).
- Tidak bisa menilai kata di luar vocabulary training (contoh nyata: *murahan* → ulasan negatif salah jadi positif).
- Perlu data berlabel yang jauh lebih banyak dan retrain berkala.

**Gemini API**
- Bergantung pada koneksi internet dan layanan eksternal. Di eksperimen ini 9 dari 80 request baru berhasil setelah dicoba ulang. Penyebabnya tidak tercatat di run tersimpan; pencatatan error retry sudah ditambahkan untuk run berikutnya.
- Model bisa deprecated atau berganti versi (terjadi pada model di starter), dan hasil harus dievaluasi ulang setiap kali berganti model.
- Konsistensi output hanya diuji dengan `temperature=0` pada 16 kalimat yang diulang (semuanya konsisten). Determinisme penuh tidak dijamin oleh API.
- Output dibatasi enum, sehingga model tidak bisa menyatakan "ragu" atau "netral".
- Isi ulasan dikirim ke server pihak ketiga. Untuk produksi, kebijakan data perlu dicek, meskipun ulasan produk umumnya bersifat publik.
- Tidak ada kesalahan prediksi yang teramati, sehingga kelemahan spesifik Gemini pada data ini belum terlihat. Ini bukan bukti bahwa Gemini bebas kesalahan.

## 8. Rekomendasi Technical Approach

**Rekomendasi: kembangkan pendekatan LLM API (Gemini, zero-shot) sebagai pendekatan utama untuk tahap berikutnya, dengan klasifikasi asinkron dan evaluasi lanjutan pada data ulasan nyata sebelum masuk produksi.**

Alasan, berdasarkan hasil eksperimen:
1. **Generalisasi ke ulasan baru.** Kebutuhan utama fitur ini adalah mengklasifikasikan ulasan yang belum pernah ada sebelumnya. Pada group split, yaitu satu-satunya evaluasi yang menguji hal ini, Gemini benar di 8/8 kalimat (40/40 baris). Scikit-learn hanya benar di 6/8 kalimat (accuracy 0,625; rata-rata 5-fold 0,643). Skor 1,000 model klasik di split utama tidak dipakai sebagai dasar keputusan karena berasal dari leakage.
2. **Data training tidak cukup.** Dengan 40 kalimat unik, model klasik tidak punya bahan untuk mempelajari variasi bahasa ulasan. Kesalahannya (negasi dan kata baru) adalah jenis kesalahan yang akan sering muncul di produksi. Memperbaikinya butuh pengumpulan dan pelabelan data dalam jumlah besar. Gemini tidak membutuhkan langkah ini.
3. **Biaya masih masuk akal.** Estimasi sekitar US$35 per 1 juta ulasan dengan tarif paid tier `gemini-3.5-flash-lite`, karena prompt-nya pendek (~102 token input, 2 token output per ulasan).
4. **Latensi bisa diatasi lewat desain sistem.** Sekitar 0,7 detik per ulasan terlalu lambat kalau dihitung saat halaman dimuat, tetapi tidak masalah kalau ulasan diklasifikasikan sekali saat dikirim pelanggan dan labelnya disimpan di database.

Syarat sebelum dipakai di produksi:
- Evaluasi ulang pada sampel ulasan nyata yang lebih besar dan beragam (termasuk sarkasme, ulasan campuran, slang, typo) dengan label dari manusia. Hasil 100% di sini belum cukup sebagai bukti.
- Jalankan lewat antrean asinkron dengan retry/backoff. Kalau API gagal, ulasan ditandai "belum diklasifikasi" dan dicoba lagi nanti, **tidak** diberi label default.
- Kunci versi model, catat versi di setiap prediksi, dan evaluasi ulang setiap kali migrasi model.
- Cek kebijakan privasi data sebelum mengirim ulasan ke API eksternal.

Kondisi yang membuat model klasik lebih cocok:
- Volume ulasan sangat besar dan biaya per request atau rate limit menjadi masalah, atau label harus muncul secara real-time tanpa ketergantungan jaringan.
- Data ulasan tidak boleh keluar dari infrastruktur internal.
- Sudah tersedia dataset berlabel yang besar dan beragam, misalnya label hasil Gemini yang sudah diverifikasi manusia, sehingga model klasik bisa dilatih ulang dan dievaluasi dengan group split. Kalau performanya mendekati Gemini, model klasik bisa menggantikan atau mendampingi Gemini (misalnya model klasik untuk kasus yang jelas, Gemini untuk kasus yang ragu). Opsi ini perlu dibuktikan dengan eksperimen terpisah dan tidak disimpulkan dari notebook ini.